# T62-文章润色 重构Notebook

## 项目概述

本项目提供一个Markdown文章润色工具，使用阿里云Qwen大模型对文章进行逐段润色，保持原有结构和风格。

## 目录

1. [环境准备与依赖安装](#1-环境准备与依赖安装)
2. [配置管理](#2-配置管理)
3. [数据加载](#3-数据加载)
4. [核心功能实现](#4-核心功能实现)
5. [执行润色任务](#5-执行润色任务)
6. [结果展示与分析](#6-结果展示与分析)
7. [总结与扩展](#7-总结与扩展)

## 1. 环境准备与依赖安装

安装必要的Python依赖包。

In [ ]:
# 安装依赖
!pip install openai markdown -q

In [ ]:
# 导入基础库
import os
import sys
from pathlib import Path

# 添加项目路径
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 导入项目模块
import config
from src.refiner import ArticleRefiner

print("环境准备完成!")
print(f"项目根目录: {project_root}")

## 2. 配置管理

查看和设置API配置。本项目使用环境变量管理敏感信息。

In [ ]:
# 显示当前配置
print("=== 当前配置 ===")
print(f"API Base URL: {config.DASHSCOPE_BASE_URL}")
print(f"模型名称: {config.MODEL_NAME}")
print(f"最大Token数: {config.MAX_TOKENS}")
print(f"温度参数: {config.TEMPERATURE}")
print(f"API Key已设置: {'是' if config.DASHSCOPE_API_KEY else '否'}")

In [ ]:
# 如果需要临时设置API Key（推荐使用环境变量）
# os.environ["DASHSCOPE_API_KEY"] = "your-api-key-here"

## 3. 数据加载

准备和加载待润色的Markdown文件。

In [ ]:
# 检查数据目录
data_dir = Path(config.DATA_DIR)
data_dir.mkdir(exist_ok=True)

print(f"数据目录: {data_dir.absolute()}")
print(f"目录中的文件:")
for f in data_dir.glob("*.md"):
    print(f"  - {f.name}")

In [ ]:
# 创建示例Markdown文件（如果不存在）
sample_file = data_dir / "sample.md"

if not sample_file.exists():
    sample_content = """# 示例文章

## 第一章 引言

这是一个关于人工智能的示例文章。人工智能技术正在快速发展，对各行各业产生深远影响。

## 第二章 技术发展

深度学习是当前人工智能领域最重要的技术之一。它通过多层神经网络来学习数据的特征表示。

自然语言处理是人工智能的重要分支，它使计算机能够理解和生成人类语言。
"""
    sample_file.write_text(sample_content, encoding='utf-8')
    print(f"已创建示例文件: {sample_file}")
else:
    print(f"示例文件已存在: {sample_file}")

In [ ]:
# 读取并预览文件内容
input_file = sample_file  # 或指定其他文件

# 初始化润色器
refiner = ArticleRefiner()
content = refiner.read_markdown_file(str(input_file))

print(f"文件: {input_file.name}")
print(f"内容长度: {len(content)} 字符")
print("\n--- 前500字符预览 ---")
print(content[:500])

## 4. 核心功能实现

展示文章润色的核心功能模块。

In [ ]:
# 段落分割演示
paragraphs = refiner.split_into_paragraphs(content)

print(f"共分割为 {len(paragraphs)} 个段落\n")
for i, p in enumerate(paragraphs, 1):
    preview = p[:100] + "..." if len(p) > 100 else p
    print(f"段落 {i}: {preview}")

In [ ]:
# 单段落润色演示（使用第一个段落）
if paragraphs:
    sample_paragraph = paragraphs[0]
    print("原始段落:")
    print(sample_paragraph)
    print("\n润色中...")
    
    refined = refiner.refine_paragraph(sample_paragraph)
    print("\n润色后:")
    print(refined)

## 5. 执行润色任务

对完整文章进行润色处理。

In [ ]:
# 设置输入输出路径
output_dir = Path(config.OUTPUT_DIR)
output_dir.mkdir(exist_ok=True)

output_file = output_dir / f"refined_{input_file.name}"

print(f"输入文件: {input_file}")
print(f"输出文件: {output_file}")

In [ ]:
# 执行润色
print("开始润色文章...")

refined_content = refiner.process_file(str(input_file), str(output_file))

print(f"\n润色完成! 已保存至: {output_file}")
print(f"润色后内容长度: {len(refined_content)} 字符")

## 6. 结果展示与分析

查看润色结果并进行对比分析。

In [ ]:
# 读取并显示润色后的内容
print("=== 润色后的文章 ===\n")
print(refined_content)

In [ ]:
# 对比分析
original_paragraphs = refiner.split_into_paragraphs(content)
refined_paragraphs = refined_content.split("\n\n")

print("=== 原文与润色对比 ===\n")
for i, (orig, refi) in enumerate(zip(original_paragraphs, refined_paragraphs), 1):
    print(f"--- 段落 {i} ---")
    print(f"原文({len(orig)}字): {orig[:80]}..." if len(orig) > 80 else f"原文({len(orig)}字): {orig}")
    print(f"润色({len(refi)}字): {refi[:80]}..." if len(refi) > 80 else f"润色({len(refi)}字): {refi}")
    print()

In [ ]:
# 统计信息
print("=== 润色统计 ===")
print(f"原始段落数: {len(original_paragraphs)}")
print(f"润色段落数: {len(refined_paragraphs)}")
print(f"原始字符数: {len(content)}")
print(f"润色字符数: {len(refined_content)}")
print(f"字符变化: {len(refined_content) - len(content):+d}")

## 7. 总结与扩展

### 项目总结

本项目实现了一个基于大语言模型的Markdown文章润色工具：

1. **模块化设计**: 将核心功能封装在`ArticleRefiner`类中
2. **环境变量管理**: 使用`os.environ.get()`安全管理API密钥
3. **逐段润色**: 保持文章原有结构和风格
4. **易于扩展**: 可轻松添加新的润色策略

### 扩展方向

1. 添加批量处理功能
2. 支持更多文档格式（Word、PDF等）
3. 添加润色风格选项（正式、口语化等）
4. 实现润色历史记录和版本对比
5. 添加进度显示和断点续传

In [ ]:
# 列出所有输出文件
print("=== 输出文件列表 ===")
for f in output_dir.glob("*"):
    print(f"  {f.name} ({f.stat().st_size} bytes)")

In [ ]:
print("Notebook执行完成!")